# Fine-tunning on Figure 3D MethylBERT data with EpigenDnabert2

This tutorial demonstrates:
1. **How to perform fine-tunning on the prepared data and save results to the local directory**.
---

### Step 0: Import needed classes and set paths

In [ ]:
from methyldl.modelling.dnabert2 import EpigenDnabert2, TrainingArguments
data_path = "../Data/Curated/MethylBERT_Figure3D"

#### Inspecting data in path 

In [ ]:
import os
import csv
for file_name in os.listdir(data_path):
    with open(os.path.join(data_path, file_name), "r") as f:
        data = list(csv.reader(f))
        pos_labels = [label for seq, meth, label in data[1:]].count("1")
        methylated_reads = len([x for x in [meth for seq, meth, label in data[1:]] if "1" in x])
        unmethylated_reads = len([x for x in [meth for seq, meth, label in data[1:]] if "0" in x and "1" not in x])
        rows = len(data)
    print(f"{file_name} has {rows} rows with {pos_labels/rows:.0%} labels positive and {methylated_reads} methylated reads and {unmethylated_reads} unmethylated reads")
    print("-"*120)

### Step 1: Perform fine-tunning

In [ ]:
model_instance = EpigenDnabert2(use_cpg_methylation=True, max_sequence_length=150)
training_args =  TrainingArguments(
            run_name = "dnabert2_default",
            per_device_train_batch_size = 300, # Batch size is set for 12Gb VRAM. Reduce proportionally to your avaliable VRAM or face consequences!!!
            per_device_eval_batch_size = 200,
            gradient_accumulation_steps = 1,
            learning_rate = 3e-5,
            fp16 = True,
            save_steps = 20,
            output_dir ="../../fine_tunning_results/MethylBERT_Figure3D/dnabert2_demo",
            evaluation_strategy = "steps",
            eval_steps = 20, 
            warmup_steps = 100, 
            logging_steps = 100, 
            num_train_epochs = 20, 
            overwrite_output_dir = True, 
            log_level = "info",
            find_unused_parameters = False,
            batch_eval_metrics = False,
            eval_and_save_results = True)
model_instance.fine_tune(data_path=data_path, training_args=training_args)